# Section 7 — Reach-and-Grasp Classification

## ⚠️ Dataset Notice

**This section requires manual download.**  
Dataset: **Jeong et al. 2020**, *IEEE Transactions on Neural Systems and Rehabilitation Engineering*,  
DOI: [10.1109/TNSRE.2020.3018788](https://doi.org/10.1109/TNSRE.2020.3018788).  

**Instructions:**
1. Download the `.mat` files from [IEEE DataPort](https://ieee-dataport.org/).
2. Place the `.mat` files in `./data/reach_grasp/`.
3. The section will skip automatically if the folder is missing.

The dataset contains 62-channel EEG recorded at 1000 Hz during a reach-and-grasp
paradigm with multiple grasp types (palmar, lateral, etc.). We classify **palmar grasp**
(class 0) vs. **lateral grasp** (class 1) — a clinically relevant binary discrimination
for stroke rehabilitation.

In [ ]:
# ============================================================================
# Section 7.0 — Dataset Availability Check
# ============================================================================
# WHY: The Jeong et al. dataset is not freely redistributable; we must check
#      for local availability before attempting any analysis.
# WHAT: Check if the reach-and-grasp data folder exists and set a flag.

import os
import sys
import warnings
import copy
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import mne
from mne.decoding import CSP

from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import (
    StratifiedKFold, GridSearchCV, cross_val_predict
)
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, classification_report,
    roc_curve, roc_auc_score, ConfusionMatrixDisplay, cohen_kappa_score
)
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

import scipy.io  # for loading .mat files from the Jeong dataset

# ── Suppress noisy MNE and deprecation warnings ──
mne.set_log_level('WARNING')
warnings.filterwarnings('ignore', category=DeprecationWarning)
sns.set_style('whitegrid')  # consistent plot style across all sections

# ── Reproducibility seeds ──
SEED = 42
np.random.seed(SEED)        # NumPy random generator seed
random.seed(SEED)           # Python stdlib random seed
torch.manual_seed(SEED)     # PyTorch CPU seed
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)         # PyTorch GPU seed (all GPUs)
    torch.backends.cudnn.deterministic = True # deterministic cuDNN ops

# ── Device selection ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ── Figure output directory ──
FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)

# ── Check dataset availability ──
REACH_GRASP_DIR = './data/reach_grasp/'  # expected location for .mat files
reach_grasp_available = os.path.exists(REACH_GRASP_DIR)  # True if folder exists

if reach_grasp_available:
    # List .mat files to confirm data is actually present
    mat_files = [f for f in os.listdir(REACH_GRASP_DIR) if f.endswith('.mat')]
    if len(mat_files) == 0:
        reach_grasp_available = False  # folder exists but is empty
        print("WARNING: ./data/reach_grasp/ exists but contains no .mat files.")
        print("Please download the Jeong et al. dataset and place .mat files there.")
    else:
        print(f"Reach-and-grasp dataset found: {len(mat_files)} .mat file(s) in {REACH_GRASP_DIR}")
        print(f"Files: {sorted(mat_files)[:5]}{'...' if len(mat_files) > 5 else ''}")
else:
    print("="*70)
    print("NOTICE: Reach-and-grasp dataset NOT found at ./data/reach_grasp/")
    print("All remaining cells in Section 7 will be SKIPPED.")
    print("To enable Section 7, download the Jeong et al. 2020 dataset from")
    print("IEEE DataPort and place .mat files in ./data/reach_grasp/")
    print("="*70)

In [ ]:
# ============================================================================
# Section 7.1 — Load Subject 1 .mat File and Inspect Structure
# ============================================================================
# WHY: The Jeong dataset stores each subject's EEG in MATLAB .mat format.
#      We need to inspect keys to understand the data layout before building
#      an MNE RawArray.
# WHAT: Load the first subject's file, print all keys, and identify EEG array,
#       channel info, and event/label arrays.

if reach_grasp_available:
    # ── Locate subject 1 file ──
    # Naming convention varies; try common patterns
    mat_files_sorted = sorted(
        [f for f in os.listdir(REACH_GRASP_DIR) if f.endswith('.mat')]
    )  # sort alphabetically so subject 1 comes first
    subj1_file = mat_files_sorted[0]  # first file = subject 1 (by convention)
    subj1_path = os.path.join(REACH_GRASP_DIR, subj1_file)
    print(f"Loading: {subj1_path}")

    # ── Load .mat file ──
    # scipy.io.loadmat returns a dict; keys prefixed with '__' are MATLAB metadata
    mat_data = scipy.io.loadmat(subj1_path)

    # ── Print all keys and their shapes ──
    # This is the critical first step: understanding what arrays are available
    print("\nKeys in .mat file:")
    for key in mat_data.keys():
        val = mat_data[key]
        if hasattr(val, 'shape'):
            print(f"  {key:30s} -> shape: {val.shape}, dtype: {val.dtype}")
        else:
            print(f"  {key:30s} -> type: {type(val).__name__}")
else:
    print("Section 7.1 SKIPPED — dataset not available.")

In [ ]:
# ============================================================================
# Section 7.2 — Build MNE RawArray from .mat Data
# ============================================================================
# WHY: MNE's RawArray is the standard container for continuous EEG. Building one
#      lets us reuse preprocess_raw() and all MNE visualization/epoching tools.
# WHAT: Extract the EEG matrix and channel names from the .mat file, create an
#       MNE Info object, and construct a RawArray.
#
# NOTE ON KEY NAVIGATION:
#   The Jeong et al. .mat structure typically contains:
#     - 'EEG' or 'eeg': (n_channels, n_samples) float64 array — raw EEG voltages
#     - 'mrk' or 'marker': event markers indicating trial onset and grasp type
#     - 'fs' or 'srate': sampling frequency (1000 Hz)
#     - 'ch_names' or embedded in header: 62 EEG channel labels (10-20 system)
#   Exact keys depend on the specific export; we use flexible detection below.

if reach_grasp_available:
    # ── Step 1: Extract sampling frequency ──
    # Look for common key names that store the sampling rate
    sfreq_keys = [k for k in mat_data.keys() if k.lower() in
                  ('fs', 'srate', 'sfreq', 'sampling_rate', 'samplingrate')]
    if sfreq_keys:
        sfreq_orig = float(mat_data[sfreq_keys[0]].flatten()[0])  # extract scalar
    else:
        sfreq_orig = 1000.0  # default per paper: 1000 Hz acquisition rate
        print("WARNING: Sampling rate key not found, defaulting to 1000 Hz (per paper).")
    print(f"Original sampling frequency: {sfreq_orig} Hz")

    # ── Step 2: Extract EEG data matrix ──
    # Convention: (n_channels, n_samples) — MNE expects this orientation
    eeg_keys = [k for k in mat_data.keys()
                if k.lower() in ('eeg', 'data', 'x', 'eegdata', 'raw_eeg')]
    if eeg_keys:
        eeg_data = mat_data[eeg_keys[0]].astype(np.float64)  # ensure float64 for MNE
    else:
        # Fallback: pick the largest numeric array (likely EEG)
        numeric_keys = [k for k in mat_data.keys()
                        if hasattr(mat_data[k], 'shape') and not k.startswith('__')]
        sizes = {k: mat_data[k].size for k in numeric_keys}
        eeg_key = max(sizes, key=sizes.get)  # largest array
        eeg_data = mat_data[eeg_key].astype(np.float64)
        print(f"WARNING: Using largest array '{eeg_key}' as EEG data.")

    # Ensure shape is (n_channels, n_samples) — transpose if needed
    if eeg_data.shape[0] > eeg_data.shape[1]:
        eeg_data = eeg_data.T  # transpose from (n_samples, n_channels) to (n_channels, n_samples)
        print("Transposed EEG data to (n_channels, n_samples) orientation.")
    print(f"EEG data shape: {eeg_data.shape}  →  {eeg_data.shape[0]} channels, {eeg_data.shape[1]} samples")

    # ── Step 3: Build channel names ──
    n_eeg_channels = eeg_data.shape[0]  # number of EEG channels
    # Try to extract channel names from .mat file
    ch_keys = [k for k in mat_data.keys()
               if 'ch' in k.lower() and 'name' in k.lower()]
    if ch_keys:
        ch_names_raw = mat_data[ch_keys[0]].flatten()
        ch_names = [str(c).strip() for c in ch_names_raw][:n_eeg_channels]
    else:
        # Generate standard 10-20 names for 62-channel montage
        ch_names = [f'EEG{i+1:03d}' for i in range(n_eeg_channels)]
        print(f"WARNING: Channel names not found, using generic labels (EEG001-EEG{n_eeg_channels:03d}).")

    # ── Step 4: Convert to Volts if data is in microvolts ──
    # MNE expects EEG in Volts; .mat files often store µV
    if np.abs(eeg_data).max() > 1.0:
        eeg_data = eeg_data * 1e-6  # convert µV → V
        print("Converted EEG data from µV to V (MNE convention).")

    # ── Step 5: Create MNE Info and RawArray ──
    info = mne.create_info(
        ch_names=ch_names,       # list of channel names
        sfreq=sfreq_orig,        # original sampling frequency (1000 Hz)
        ch_types='eeg'           # all channels are EEG type
    )  # Info object holds metadata: channel names, types, sfreq, montage

    raw_rg = mne.io.RawArray(
        eeg_data, info
    )  # RawArray wraps the numpy array into an MNE Raw object

    # Try to set a standard montage for topographic plotting
    try:
        montage = mne.channels.make_standard_montage('standard_1020')
        raw_rg.set_montage(montage, on_missing='warn')
    except Exception:
        print("WARNING: Could not set standard 10-20 montage (custom channel names).")

    print(f"\nMNE RawArray created: {raw_rg}")
    print(f"  Duration: {raw_rg.times[-1]:.1f} s")
    print(f"  Channels: {len(raw_rg.ch_names)}")
    print(f"  Sfreq: {raw_rg.info['sfreq']} Hz")
else:
    print("Section 7.2 SKIPPED — dataset not available.")

In [ ]:
# ============================================================================
# Section 7.3 — Preprocess: Band-pass Filter and Resample to 250 Hz
# ============================================================================
# WHY RESAMPLING FROM 1000 Hz TO 250 Hz IS VALID:
#   By the Nyquist theorem, a signal sampled at fs can faithfully represent
#   frequencies up to fs/2. Our analysis band is 8–30 Hz (mu + beta rhythms).
#   At 250 Hz the Nyquist frequency is 125 Hz — well above 30 Hz — so NO
#   information in our band of interest is lost. Downsampling by 4× reduces:
#     • Memory: 4× fewer samples per trial
#     • Compute: convolutions / CSP / attention scale with n_timepoints
#     • Noise: anti-aliasing filter attenuates high-frequency EMG artifacts
#   This also matches the BCI Competition IV 2a dataset (250 Hz), enabling
#   more direct cross-dataset comparisons.
#
# WHAT: Apply preprocess_raw() with sfreq_target=250. Since the reach-grasp
#       dataset uses different event codes, we handle epoching separately.

if reach_grasp_available:
    # ── Apply band-pass filter (8-30 Hz) ──
    # IIR Butterworth, order=5: steep roll-off, minimal phase distortion
    # 8 Hz high-pass removes slow drifts + delta/theta; 30 Hz low-pass removes
    # muscle artifacts (EMG) and line noise harmonics
    raw_rg.load_data()  # ensure data is in memory for filtering
    raw_rg.filter(
        l_freq=8.0,   # lower cutoff: 8 Hz (mu band onset)
        h_freq=30.0,  # upper cutoff: 30 Hz (beta band ceiling)
        method='iir',
        iir_params={'order': 5, 'ftype': 'butter'}  # 5th-order Butterworth
    )
    print("Band-pass filtered: 8–30 Hz (IIR Butterworth order 5)")

    # ── Resample from 1000 Hz to 250 Hz ──
    # MNE's resample() applies an anti-aliasing FIR filter before decimation
    # to prevent spectral aliasing below the new Nyquist frequency (125 Hz)
    raw_rg.resample(sfreq=250.0)  # downsample by factor of 4
    print(f"Resampled to {raw_rg.info['sfreq']} Hz (Nyquist = {raw_rg.info['sfreq']/2} Hz)")

    # ── Apply average reference ──
    # Re-references all channels to the mean across electrodes,
    # removing global noise shared by all channels (e.g., line noise)
    raw_rg.set_eeg_reference('average', projection=True)
    raw_rg.apply_proj()  # apply the average reference projection
    print("Average reference applied.")

    print(f"\nPreprocessed RawArray: {raw_rg.info['sfreq']} Hz, "
          f"{len(raw_rg.ch_names)} channels, "
          f"{raw_rg.times[-1]:.1f} s duration")
else:
    print("Section 7.3 SKIPPED — dataset not available.")

In [ ]:
# ============================================================================
# Section 7.4 — Extract Events, Epoch, and Build Binary Labels
# ============================================================================
# WHY: We need discrete trial epochs aligned to movement onset, with binary
#      labels (palmar=0, lateral=1) for supervised classification.
# WHAT: Parse event markers from the .mat file, create MNE events array,
#       epoch [0, 3]s post-movement-onset, and verify class balance.

if reach_grasp_available:
    # ── Step 1: Extract event/marker information ──
    # The Jeong dataset encodes grasp type via event markers in the continuous data.
    # Common keys: 'mrk', 'marker', 'trigger', 'event', 'label', 'y'
    mrk_keys = [k for k in mat_data.keys()
                if any(tag in k.lower() for tag in
                       ('mrk', 'marker', 'trigger', 'event', 'label', 'y', 'class'))
                and not k.startswith('__')]
    print(f"Potential event/label keys: {mrk_keys}")

    # ── Step 2: Parse event onset times and class labels ──
    # Strategy: look for separate onset + label arrays, or a combined struct
    events_list = []  # will hold (sample_idx, 0, event_id) tuples

    if len(mrk_keys) >= 1:
        # Try to extract labels array
        label_data = None
        onset_data = None
        for k in mrk_keys:
            arr = mat_data[k]
            if hasattr(arr, 'flatten'):
                flat = arr.flatten()
                # If values are small integers (1-6), likely class labels
                unique_vals = np.unique(flat[flat != 0])  # exclude zero (no event)
                if len(unique_vals) <= 10 and np.all(unique_vals == unique_vals.astype(int)):
                    if label_data is None or flat.size < label_data.size:
                        label_data = flat
                        print(f"  Labels from '{k}': unique values = {unique_vals}")

        # If labels are embedded in a continuous marker channel (same length as EEG),
        # find transitions (rising edges) as event onsets
        if label_data is not None and label_data.size > 1000:
            # This is a continuous marker channel — detect onset of each trial
            # Account for resampling: marker indices are at original sfreq
            resample_ratio = 250.0 / sfreq_orig  # scaling factor for sample indices
            diff = np.diff(label_data.astype(int))  # detect transitions
            onset_idx = np.where(diff > 0)[0] + 1   # +1: index of first nonzero sample
            event_codes = label_data[onset_idx].astype(int)  # class code at each onset
            # Scale indices to resampled time base
            onset_idx_resampled = (onset_idx * resample_ratio).astype(int)
            for idx, code in zip(onset_idx_resampled, event_codes):
                events_list.append([idx, 0, code])  # MNE events format: [sample, 0, id]
            print(f"  Found {len(events_list)} events via marker transitions")

        elif label_data is not None and label_data.size <= 1000:
            # Labels per trial (not continuous) — need separate onset info
            # Look for onset/time arrays
            for k in mrk_keys:
                arr = mat_data[k]
                if hasattr(arr, 'flatten'):
                    flat = arr.flatten()
                    if flat.size == label_data.size and np.all(flat >= 0):
                        if np.max(flat) > 100:  # likely sample indices
                            onset_data = flat
                            break
            if onset_data is not None:
                resample_ratio = 250.0 / sfreq_orig
                for onset, lbl in zip(onset_data, label_data):
                    sample_idx = int(onset * resample_ratio)
                    events_list.append([sample_idx, 0, int(lbl)])
            else:
                # Assume equal spacing (fallback)
                n_trials = label_data.size
                total_samples = raw_rg.n_times
                spacing = total_samples // (n_trials + 1)
                for i, lbl in enumerate(label_data):
                    events_list.append([(i + 1) * spacing, 0, int(lbl)])
                print(f"  WARNING: Using estimated equal-spaced trial onsets ({spacing} samples apart)")

    # ── Fallback: use MNE's find_events on a stim channel if present ──
    if len(events_list) == 0:
        try:
            events_mne = mne.find_events(raw_rg, shortest_event=1)
            events_list = events_mne.tolist()
            print(f"  Found {len(events_list)} events via mne.find_events()")
        except Exception:
            print("ERROR: Could not extract events from the dataset.")
            print("Please check the .mat file structure and adjust key names above.")
            reach_grasp_available = False  # abort remaining cells

    if reach_grasp_available and len(events_list) > 0:
        events_array = np.array(events_list, dtype=int)  # (n_events, 3)
        unique_events = np.unique(events_array[:, 2])
        print(f"\nEvent codes found: {unique_events}")
        for code in unique_events:
            count = np.sum(events_array[:, 2] == code)
            print(f"  Code {code}: {count} trials")

        # ── Step 3: Map to binary labels ──
        # Jeong et al.: palmar grasp and lateral grasp are two of the grasp types
        # We select the two relevant event codes (typically 1=palmar, 2=lateral,
        # but codes vary by export version — we take the first two unique codes)
        PALMAR_CODE = unique_events[0]   # palmar grasp → class 0
        LATERAL_CODE = unique_events[1]  # lateral grasp → class 1
        event_id = {'palmar_grasp': int(PALMAR_CODE),
                     'lateral_grasp': int(LATERAL_CODE)}
        print(f"\nBinary mapping: palmar_grasp (code {PALMAR_CODE}) = class 0, "
              f"lateral_grasp (code {LATERAL_CODE}) = class 1")

        # ── Step 4: Epoch [0, 3]s post-movement-onset ──
        # 3-second window captures the reach + grasp execution phase
        # tmin=0: start at movement onset (no pre-stimulus baseline needed for
        # reach-grasp — the relevant motor activity begins at onset)
        epochs_rg = mne.Epochs(
            raw_rg,
            events=events_array,
            event_id=event_id,        # only keep palmar + lateral trials
            tmin=0.0,                 # start at movement onset
            tmax=3.0,                 # 3s post-onset window
            baseline=None,            # no baseline correction (onset = t=0)
            preload=True,             # load all epoch data into RAM
            reject=dict(eeg=200e-6),  # reject epochs with peak-to-peak > 200 µV
            on_missing='warn'         # warn (don't crash) if some events are outside data
        )

        # ── Step 5: Extract numpy arrays and build binary labels ──
        X_rg = epochs_rg.get_data()     # (n_epochs, n_channels, n_timepoints)
        y_rg_raw = epochs_rg.events[:, 2]  # raw event codes
        # Map: palmar → 0, lateral → 1
        y_rg = np.where(y_rg_raw == PALMAR_CODE, 0, 1).astype(np.int64)

        n_palmar = np.sum(y_rg == 0)   # count class 0 (palmar)
        n_lateral = np.sum(y_rg == 1)  # count class 1 (lateral)

        print(f"\nEpoch dimensions: {X_rg.shape}")
        print(f"  n_epochs     = {X_rg.shape[0]}")
        print(f"  n_channels   = {X_rg.shape[1]}")
        print(f"  n_timepoints = {X_rg.shape[2]}")
        print(f"\nClass balance:")
        print(f"  Palmar grasp  (class 0): {n_palmar} epochs")
        print(f"  Lateral grasp (class 1): {n_lateral} epochs")
        print(f"  Ratio: {n_palmar / max(n_lateral, 1):.2f}")

        # Warn if imbalanced (>2:1 ratio)
        if max(n_palmar, n_lateral) / max(min(n_palmar, n_lateral), 1) > 2:
            print("  WARNING: Classes are imbalanced (>2:1). Consider balancing strategies.")
    else:
        print("No events extracted — cannot proceed with epoching.")
        reach_grasp_available = False
else:
    print("Section 7.4 SKIPPED — dataset not available.")

---
## 7.5 — CSP + SVM Classification (Reach-and-Grasp)

In [ ]:
# ============================================================================
# Section 7.5 — CSP + SVM Pipeline for Reach-and-Grasp
# ============================================================================
# WHY: CSP is the gold-standard spatial filter for two-class motor imagery BCI.
#      It maximizes variance ratio between classes — if palmar/lateral grasp
#      produce distinguishable spatial patterns, CSP will find them.
# WHAT: Same nested-CV pipeline as Section 4 (5-fold outer, 3-fold inner GridSearch)
#       adapted to the reach-grasp epoch data.

if reach_grasp_available:
    print("Running CSP + SVM pipeline on reach-and-grasp data...")
    print(f"  Input shape: X={X_rg.shape}, y={y_rg.shape}")

    # ── Define the CSP → SVM pipeline ──
    # CSP: 6 components (3 per class) — more than the 4 used for left/right MI
    # because grasp types have more complex spatial patterns across motor cortex
    pipe_rg = Pipeline([
        ('csp', CSP(n_components=6, log=True, reg=None, norm_trace=True)),
        ('svm', SVC(kernel='rbf', probability=True, random_state=SEED))
    ])

    # ── Hyperparameter grid for inner CV ──
    param_grid_rg = {
        'svm__C': [0.1, 1, 10, 100],    # regularization strength (lower = more regularized)
        'svm__gamma': ['scale', 'auto']  # RBF kernel width
    }

    # ── Outer CV: 5-fold stratified ──
    outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    # ── Inner CV: 3-fold GridSearchCV for hyperparameter tuning ──
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    grid_search_rg = GridSearchCV(
        pipe_rg,
        param_grid_rg,
        cv=inner_cv,        # 3-fold inner CV
        scoring='accuracy', # optimize for accuracy
        n_jobs=-1,          # use all CPU cores for parallelism
        refit=True          # refit best model on full inner training set
    )

    # ── Nested cross-validation ──
    # cross_val_predict gives out-of-fold predictions for every sample
    y_pred_csp_rg = cross_val_predict(
        grid_search_rg, X_rg, y_rg,
        cv=outer_cv,
        method='predict'
    )

    # ── Compute metrics ──
    acc_csp_rg = accuracy_score(y_rg, y_pred_csp_rg)  # overall accuracy
    f1_csp_rg = f1_score(y_rg, y_pred_csp_rg, average='binary')  # binary F1
    kappa_csp_rg = cohen_kappa_score(y_rg, y_pred_csp_rg)  # agreement beyond chance

    print(f"\n{'='*50}")
    print(f"CSP + SVM Results (Reach-and-Grasp, Subject 1)")
    print(f"{'='*50}")
    print(f"  Accuracy:      {acc_csp_rg:.4f} ({acc_csp_rg*100:.1f}%)")
    print(f"  F1 Score:      {f1_csp_rg:.4f}")
    print(f"  Cohen's Kappa: {kappa_csp_rg:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(
        y_rg, y_pred_csp_rg,
        target_names=['Palmar Grasp', 'Lateral Grasp']
    ))

    # ── Confusion Matrix Plot ──
    fig, ax = plt.subplots(1, 1, figsize=(6, 5))
    cm_csp_rg = confusion_matrix(y_rg, y_pred_csp_rg, normalize='true')  # row-normalized
    sns.heatmap(
        cm_csp_rg, annot=True, fmt='.2f', cmap='Blues',
        xticklabels=['Palmar', 'Lateral'],
        yticklabels=['Palmar', 'Lateral'],
        ax=ax
    )
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('True', fontsize=12)
    ax.set_title(f'CSP+SVM Confusion Matrix\nAcc={acc_csp_rg:.3f}, F1={f1_csp_rg:.3f}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'section7_fig1_csp_svm_confusion.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Section 7.5 SKIPPED — dataset not available.")

### What to look for in this plot:
- **Diagonal dominance**: high values on the diagonal indicate correct classification. If off-diagonal cells are large, the model confuses the two grasp types.
- **Asymmetric errors**: if one grasp type is systematically misclassified as the other, it suggests the motor cortex patterns overlap more in one direction.
- **Comparison to left/right MI**: expect lower accuracy here because palmar vs. lateral grasp share more cortical territory (both activate contralateral motor cortex, differing mainly in fine-grained somatotopic detail).

---
## 7.6 — EEGNet Classification (Reach-and-Grasp)

In [ ]:
# ============================================================================
# Section 7.6 — EEGNet for Reach-and-Grasp (Train from Scratch)
# ============================================================================
# WHY: EEGNet is a compact CNN designed for EEG BCI. Training from scratch on
#      the reach-grasp data tests whether learned spatiotemporal filters can
#      capture the finer-grained cortical differences between grasp types.
# WHAT: Instantiate EEGNet with n_channels=62 (adapted to Jeong dataset),
#       5-fold stratified CV, report accuracy and F1.

if reach_grasp_available:
    print("Training EEGNet on reach-and-grasp data...")

    # ── Z-score normalize each epoch per channel ──
    # WHY: Removes absolute amplitude differences across trials and channels,
    # so the network focuses on relative temporal patterns
    eps = 1e-8  # numerical stability constant
    X_rg_norm = X_rg.copy()  # don't modify original
    mean_rg = X_rg_norm.mean(axis=2, keepdims=True)   # mean over time dimension
    std_rg = X_rg_norm.std(axis=2, keepdims=True)     # std over time dimension
    X_rg_norm = (X_rg_norm - mean_rg) / (std_rg + eps)  # z-score normalize

    n_ch_rg = X_rg_norm.shape[1]    # number of channels (62)
    n_tp_rg = X_rg_norm.shape[2]    # number of timepoints (750 @ 250 Hz, 3s window)

    # ── 5-fold stratified CV ──
    outer_cv_eegnet = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_accs_eegnet_rg = []   # store per-fold accuracy
    fold_f1s_eegnet_rg = []    # store per-fold F1
    all_preds_eegnet_rg = np.zeros_like(y_rg)  # out-of-fold predictions

    for fold_i, (train_idx, test_idx) in enumerate(outer_cv_eegnet.split(X_rg_norm, y_rg)):
        print(f"  Fold {fold_i+1}/5", end=' ')

        # ── Split train into train + validation (85/15) ──
        n_train = len(train_idx)
        n_val = max(int(0.15 * n_train), 1)  # 15% of training set for validation
        # Shuffle training indices and split
        rng = np.random.RandomState(SEED + fold_i)  # fold-specific seed
        shuffled = rng.permutation(train_idx)
        val_idx = shuffled[:n_val]        # validation subset
        train_idx_inner = shuffled[n_val:]  # training subset

        # ── Build DataLoaders ──
        # Add conv2d channel dim: (N, C, T) → (N, 1, C, T)
        X_train_t = torch.FloatTensor(X_rg_norm[train_idx_inner][:, np.newaxis, :, :])
        y_train_t = torch.LongTensor(y_rg[train_idx_inner])
        X_val_t = torch.FloatTensor(X_rg_norm[val_idx][:, np.newaxis, :, :])
        y_val_t = torch.LongTensor(y_rg[val_idx])
        X_test_t = torch.FloatTensor(X_rg_norm[test_idx][:, np.newaxis, :, :])
        y_test_t = torch.LongTensor(y_rg[test_idx])

        train_loader = DataLoader(
            TensorDataset(X_train_t, y_train_t),
            batch_size=32, shuffle=True  # shuffle each epoch for SGD
        )
        val_loader = DataLoader(
            TensorDataset(X_val_t, y_val_t),
            batch_size=32, shuffle=False
        )
        test_loader = DataLoader(
            TensorDataset(X_test_t, y_test_t),
            batch_size=32, shuffle=False
        )

        # ── Instantiate EEGNet ──
        # n_channels=62 (Jeong dataset), kernel_length=64 (~256 ms @ 250 Hz)
        model_eegnet_rg = EEGNet(
            n_classes=2,
            n_channels=n_ch_rg,         # 62 channels
            n_timepoints=n_tp_rg,       # timepoints in 3s window
            F1=8, D=2, F2=16,           # standard EEGNet hyperparameters
            dropout_rate=0.5,
            kernel_length=64            # temporal kernel: ~256 ms at 250 Hz
        ).to(device)

        # ── Train with early stopping ──
        model_eegnet_rg, history_rg, best_epoch_rg = train_eegnet(
            model_eegnet_rg, train_loader, val_loader,
            n_epochs=300, lr=0.001, patience=20, device=str(device)
        )

        # ── Evaluate on test fold ──
        model_eegnet_rg.eval()  # disable dropout for inference
        fold_preds = []
        with torch.no_grad():  # no gradient computation during evaluation
            for X_batch, _ in test_loader:
                X_batch = X_batch.to(device)
                logits = model_eegnet_rg(X_batch)
                preds = logits.argmax(dim=1).cpu().numpy()  # predicted class
                fold_preds.append(preds)

        fold_preds = np.concatenate(fold_preds)
        all_preds_eegnet_rg[test_idx] = fold_preds  # store out-of-fold predictions

        fold_acc = accuracy_score(y_rg[test_idx], fold_preds)
        fold_f1 = f1_score(y_rg[test_idx], fold_preds, average='binary')
        fold_accs_eegnet_rg.append(fold_acc)
        fold_f1s_eegnet_rg.append(fold_f1)
        print(f"→ Acc={fold_acc:.3f}, F1={fold_f1:.3f} (best epoch: {best_epoch_rg})")

    # ── Overall metrics ──
    acc_eegnet_rg = accuracy_score(y_rg, all_preds_eegnet_rg)
    f1_eegnet_rg = f1_score(y_rg, all_preds_eegnet_rg, average='binary')
    kappa_eegnet_rg = cohen_kappa_score(y_rg, all_preds_eegnet_rg)

    print(f"\n{'='*50}")
    print(f"EEGNet Results (Reach-and-Grasp, Subject 1)")
    print(f"{'='*50}")
    print(f"  Accuracy:      {acc_eegnet_rg:.4f} ({acc_eegnet_rg*100:.1f}%)")
    print(f"  F1 Score:      {f1_eegnet_rg:.4f}")
    print(f"  Cohen's Kappa: {kappa_eegnet_rg:.4f}")
    print(f"  Per-fold Acc:  {[f'{a:.3f}' for a in fold_accs_eegnet_rg]}")
    print(f"\nClassification Report:")
    print(classification_report(
        y_rg, all_preds_eegnet_rg,
        target_names=['Palmar Grasp', 'Lateral Grasp']
    ))
else:
    print("Section 7.6 SKIPPED — dataset not available.")

In [ ]:
# ============================================================================
# Section 7.6b — EEGNet Confusion Matrix (Reach-and-Grasp)
# ============================================================================
# WHAT: Visualize EEGNet classification performance as a normalized confusion matrix.

if reach_grasp_available:
    fig, ax = plt.subplots(1, 1, figsize=(6, 5))
    cm_eegnet_rg = confusion_matrix(y_rg, all_preds_eegnet_rg, normalize='true')
    sns.heatmap(
        cm_eegnet_rg, annot=True, fmt='.2f', cmap='Blues',
        xticklabels=['Palmar', 'Lateral'],
        yticklabels=['Palmar', 'Lateral'],
        ax=ax
    )
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('True', fontsize=12)
    ax.set_title(f'EEGNet Confusion Matrix (Reach-and-Grasp)\n'
                 f'Acc={acc_eegnet_rg:.3f}, F1={f1_eegnet_rg:.3f}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'section7_fig2_eegnet_confusion.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Section 7.6b SKIPPED — dataset not available.")

### What to look for in this plot:
- **Comparison to CSP+SVM**: Does EEGNet improve over CSP? Deep learning excels when spatial patterns are complex and non-stationary — both properties of reach-and-grasp EEG.
- **Per-class performance**: If one grasp type is consistently harder, this suggests overlapping cortical representations for that movement.
- **Diagonal values near 0.5**: would indicate chance-level — the model cannot distinguish the two grasp types. Values above 0.6 suggest meaningful cortical differences exist.

---
## 7.7 — DSCMI Classification (Reach-and-Grasp)

In [ ]:
# ============================================================================
# Section 7.7 — DSCMI (Depthwise Separable Conv + Multi-head Sparse Attention)
#               for Reach-and-Grasp Classification
# ============================================================================
# WHY: DSCMINet (Fu et al. 2025) adds sparse self-attention on top of depthwise
#      separable convolutions. The attention mechanism can capture long-range
#      temporal dependencies in the 3s grasp window — potentially useful for
#      reach-and-grasp where the motor plan unfolds over time.
# WHAT: Train DSCMINet from scratch, 5-fold stratified CV, report acc + F1.

if reach_grasp_available:
    print("Training DSCMI on reach-and-grasp data...")

    # ── 5-fold stratified CV ──
    outer_cv_dscmi = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_accs_dscmi_rg = []   # per-fold accuracy
    fold_f1s_dscmi_rg = []    # per-fold F1
    all_preds_dscmi_rg = np.zeros_like(y_rg)  # out-of-fold predictions

    for fold_i, (train_idx, test_idx) in enumerate(outer_cv_dscmi.split(X_rg_norm, y_rg)):
        print(f"  Fold {fold_i+1}/5", end=' ')

        # ── Split train into train + validation (85/15) ──
        n_train = len(train_idx)
        n_val = max(int(0.15 * n_train), 1)  # 15% for early stopping validation
        rng = np.random.RandomState(SEED + fold_i)  # fold-specific seed
        shuffled = rng.permutation(train_idx)
        val_idx = shuffled[:n_val]
        train_idx_inner = shuffled[n_val:]

        # ── Build DataLoaders (with conv2d channel dim) ──
        X_train_t = torch.FloatTensor(X_rg_norm[train_idx_inner][:, np.newaxis, :, :])
        y_train_t = torch.LongTensor(y_rg[train_idx_inner])
        X_val_t = torch.FloatTensor(X_rg_norm[val_idx][:, np.newaxis, :, :])
        y_val_t = torch.LongTensor(y_rg[val_idx])
        X_test_t = torch.FloatTensor(X_rg_norm[test_idx][:, np.newaxis, :, :])
        y_test_t = torch.LongTensor(y_rg[test_idx])

        train_loader_dscmi = DataLoader(
            TensorDataset(X_train_t, y_train_t),
            batch_size=32, shuffle=True
        )
        val_loader_dscmi = DataLoader(
            TensorDataset(X_val_t, y_val_t),
            batch_size=32, shuffle=False
        )
        test_loader_dscmi = DataLoader(
            TensorDataset(X_test_t, y_test_t),
            batch_size=32, shuffle=False
        )

        # ── Instantiate DSCMINet ──
        # n_channels=62 (Jeong dataset), n_timepoints adapted to 3s @ 250 Hz
        # n_heads=4 (fewer than 8 — smaller dataset benefits from less complex attention)
        # c=0.5 (sparse coefficient — balance between full and very sparse attention)
        model_dscmi_rg = DSCMINet(
            n_channels=n_ch_rg,       # 62 EEG channels
            n_timepoints=n_tp_rg,     # timepoints in 3s window
            n_classes=2,              # palmar vs. lateral
            n_heads=4,                # attention heads (moderate complexity)
            c=0.5,                    # sparse attention coefficient
            F1=16,                    # temporal filters in block 1
            D=1,                      # depth multiplier
            F2=16,                    # pointwise filters in block 3
            dropout=0.25              # dropout probability
        ).to(device)

        # ── Train with early stopping ──
        # train_model() uses Adam + ReduceLROnPlateau + early stopping (patience=30)
        history_dscmi_rg, _ = train_model(
            model_dscmi_rg, train_loader_dscmi, val_loader_dscmi, device,
            n_epochs=300, lr=1e-3, patience=30
        )

        # ── Evaluate on test fold ──
        model_dscmi_rg.eval()
        fold_preds = []
        with torch.no_grad():
            for X_batch, _ in test_loader_dscmi:
                X_batch = X_batch.to(device)
                logits = model_dscmi_rg(X_batch)
                preds = logits.argmax(dim=1).cpu().numpy()
                fold_preds.append(preds)

        fold_preds = np.concatenate(fold_preds)
        all_preds_dscmi_rg[test_idx] = fold_preds

        fold_acc = accuracy_score(y_rg[test_idx], fold_preds)
        fold_f1 = f1_score(y_rg[test_idx], fold_preds, average='binary')
        fold_accs_dscmi_rg.append(fold_acc)
        fold_f1s_dscmi_rg.append(fold_f1)
        print(f"→ Acc={fold_acc:.3f}, F1={fold_f1:.3f}")

    # ── Overall metrics ──
    acc_dscmi_rg = accuracy_score(y_rg, all_preds_dscmi_rg)
    f1_dscmi_rg = f1_score(y_rg, all_preds_dscmi_rg, average='binary')
    kappa_dscmi_rg = cohen_kappa_score(y_rg, all_preds_dscmi_rg)

    print(f"\n{'='*50}")
    print(f"DSCMI Results (Reach-and-Grasp, Subject 1)")
    print(f"{'='*50}")
    print(f"  Accuracy:      {acc_dscmi_rg:.4f} ({acc_dscmi_rg*100:.1f}%)")
    print(f"  F1 Score:      {f1_dscmi_rg:.4f}")
    print(f"  Cohen's Kappa: {kappa_dscmi_rg:.4f}")
    print(f"  Per-fold Acc:  {[f'{a:.3f}' for a in fold_accs_dscmi_rg]}")
    print(f"\nClassification Report:")
    print(classification_report(
        y_rg, all_preds_dscmi_rg,
        target_names=['Palmar Grasp', 'Lateral Grasp']
    ))
else:
    print("Section 7.7 SKIPPED — dataset not available.")

In [ ]:
# ============================================================================
# Section 7.7b — DSCMI Confusion Matrix (Reach-and-Grasp)
# ============================================================================

if reach_grasp_available:
    fig, ax = plt.subplots(1, 1, figsize=(6, 5))
    cm_dscmi_rg = confusion_matrix(y_rg, all_preds_dscmi_rg, normalize='true')
    sns.heatmap(
        cm_dscmi_rg, annot=True, fmt='.2f', cmap='Blues',
        xticklabels=['Palmar', 'Lateral'],
        yticklabels=['Palmar', 'Lateral'],
        ax=ax
    )
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('True', fontsize=12)
    ax.set_title(f'DSCMI Confusion Matrix (Reach-and-Grasp)\n'
                 f'Acc={acc_dscmi_rg:.3f}, F1={f1_dscmi_rg:.3f}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'section7_fig3_dscmi_confusion.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Section 7.7b SKIPPED — dataset not available.")

### What to look for in this plot:
- **Sparse attention benefit**: DSCMI's multi-head sparse self-attention should capture long-range temporal dependencies in the 3s grasp window. Compare diagonal values to EEGNet — if DSCMI improves, it means temporal context matters for distinguishing grasp types.
- **Overfitting risk**: DSCMI has more parameters than EEGNet. If confusion matrix values are similar or worse, the model may overfit on the smaller reach-grasp dataset.
- **Class-specific improvement**: check whether the attention mechanism helps more for one grasp type than the other.

In [ ]:
# ============================================================================
# Section 7.8 — Model Comparison Summary (Reach-and-Grasp)
# ============================================================================
# WHY: Side-by-side comparison reveals which approach best captures the fine
#      cortical differences between grasp types.
# WHAT: Bar chart comparing accuracy and F1 across all three models.

if reach_grasp_available:
    # ── Build comparison dataframe ──
    comparison_data = {
        'Model': ['CSP+SVM', 'EEGNet', 'DSCMI'],
        'Accuracy': [acc_csp_rg, acc_eegnet_rg, acc_dscmi_rg],
        'F1 Score': [f1_csp_rg, f1_eegnet_rg, f1_dscmi_rg],
        'Kappa': [kappa_csp_rg, kappa_eegnet_rg, kappa_dscmi_rg]
    }
    df_compare = pd.DataFrame(comparison_data)
    print("Model Comparison (Reach-and-Grasp, Subject 1):")
    print(df_compare.to_string(index=False, float_format='{:.4f}'.format))

    # ── Bar chart ──
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    colors = ['#2196F3', '#4CAF50', '#FF9800']  # blue, green, orange

    # Accuracy bars
    bars1 = axes[0].bar(
        df_compare['Model'], df_compare['Accuracy'],
        color=colors, edgecolor='black', linewidth=0.8
    )
    axes[0].axhline(y=0.5, color='red', linestyle='--', linewidth=1.5,
                    label='Chance (50%)')  # chance level for binary classification
    axes[0].set_ylabel('Accuracy', fontsize=12)
    axes[0].set_title('Accuracy Comparison', fontsize=14, fontweight='bold')
    axes[0].set_ylim(0, 1.0)
    axes[0].legend()
    # Add value labels on bars
    for bar, val in zip(bars1, df_compare['Accuracy']):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                     f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

    # F1 bars
    bars2 = axes[1].bar(
        df_compare['Model'], df_compare['F1 Score'],
        color=colors, edgecolor='black', linewidth=0.8
    )
    axes[1].axhline(y=0.5, color='red', linestyle='--', linewidth=1.5,
                    label='Chance (50%)')
    axes[1].set_ylabel('F1 Score', fontsize=12)
    axes[1].set_title('F1 Score Comparison', fontsize=14, fontweight='bold')
    axes[1].set_ylim(0, 1.0)
    axes[1].legend()
    for bar, val in zip(bars2, df_compare['F1 Score']):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                     f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

    plt.suptitle('Reach-and-Grasp: Model Comparison (Subject 1)',
                 fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'section7_fig4_model_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Section 7.8 SKIPPED — dataset not available.")

### What to look for in this plot:
- **All models above chance (0.5)**: any bar below the red dashed line means the model is at or below random guessing — the grasp types may not be separable with that method.
- **Deep learning vs. CSP**: if EEGNet or DSCMI significantly outperform CSP+SVM, it suggests that nonlinear spatiotemporal features are important for grasp discrimination.
- **DSCMI vs. EEGNet**: the attention mechanism in DSCMI may or may not help — the reach-grasp dataset is relatively small, which may favor the more parameter-efficient EEGNet.
- **Absolute accuracy level**: expect 55–75% for this harder binary task (vs. 70–90% for left/right MI), reflecting the greater difficulty of distinguishing grasp types.

---
## 7.9 — Discussion: Reach-and-Grasp vs. Left/Right Motor Imagery

### Why reach-and-grasp classification is harder than left/right MI

Left vs. right motor imagery produces well-separated cortical patterns: left-hand imagery
generates event-related desynchronization (ERD) over the **right** motor cortex (contralateral C4),
and vice versa. This large-scale **inter-hemispheric contrast** is the reason CSP achieves >80%
accuracy on many subjects. In contrast, palmar and lateral grasp both involve the **same hand**,
activating largely overlapping regions of the contralateral primary motor cortex (M1). The
cortical differences between grasp types depend on **fine somatotopic organization** — subtle
variations in which finger/muscle representations are recruited. These differences are
smaller in spatial extent, more variable across trials (due to co-contraction patterns,
wrist posture adjustments, and variable grasp force), and more susceptible to noise from
muscle artifacts (EMG). Additionally, reach-and-grasp involves **multi-joint coordination**
(shoulder, elbow, wrist, fingers), increasing neural variability compared to the
relatively isolated single-limb imagery in standard MI paradigms.

### Why reach-and-grasp is more clinically relevant for ADL rehabilitation

Despite the classification challenge, reach-and-grasp is **far more clinically relevant**
than left/right MI for stroke rehabilitation. The primary goal of upper-limb rehab is
restoring **functional grasping** — the ability to pick up a cup, turn a doorknob, or
use utensils. These are **activities of daily living (ADL)** that directly impact patient
independence. A BCI that can distinguish grasp types enables task-specific neurofeedback:
patients practice the *exact* motor patterns they need to recover, rather than abstract
left/right imagery that has limited transfer to real-world hand function. This makes
the harder classification problem a worthwhile engineering challenge with direct
translational value.

In [ ]:
# ============================================================================
# Section 7 — Completion Summary
# ============================================================================

if reach_grasp_available:
    print(f"Section 7 complete -- Reach-and-grasp classification: "
          f"CSP+SVM={acc_csp_rg:.1%}, EEGNet={acc_eegnet_rg:.1%}, "
          f"DSCMI={acc_dscmi_rg:.1%}")
else:
    print("Section 7 complete -- SKIPPED (dataset not available at ./data/reach_grasp/)")

---
---

# Section 8 — From BCI Model to Virtual Arm: Unity Integration Roadmap

This section outlines the full pipeline for deploying the trained EEG classification
models into a **Unity 3D environment** to drive a virtual arm in real time — the
ultimate goal of this project.

---

## 8.1 — Architecture Overview

The real-time BCI-to-Unity system consists of three loosely coupled components
connected via a local network socket (TCP or LSL):

```
┌──────────────────┐       ┌──────────────────┐       ┌──────────────────────┐
│  EEG Acquisition │  ──►  │  Python Backend   │  ──►  │   Unity Application  │
│  (OpenBCI / g.tec│       │  (Inference Server)│       │   (Virtual Arm)      │
│   / LSL stream)  │       │                    │       │                      │
└──────────────────┘       └──────────────────┘       └──────────────────────┘
     64-ch EEG               Classification              Arm animation
     @ 250 Hz                 every ~500 ms               left / right / grasp
```

1. **EEG Acquisition**: A headset (e.g., OpenBCI Cyton 16-ch, g.tec g.USBamp 64-ch, or
   any LSL-compatible device) streams raw EEG to the Python backend via
   [Lab Streaming Layer (LSL)](https://github.com/sccn/labstreaminglayer).

2. **Python Inference Server**: A lightweight Python script receives the stream,
   applies the same preprocessing pipeline used in training (band-pass 8–30 Hz,
   average reference, optional ICA), feeds sliding windows to the trained model
   (EEGNet or DSCMI exported via TorchScript), and sends the predicted class
   label + confidence score to Unity over a TCP socket.

3. **Unity Application**: A C# script receives predictions and maps them to
   virtual arm animations (e.g., left reach, right reach, palmar grasp,
   lateral grasp) via Unity's Animator Controller.

---

## 8.2 — Step 1: Export the Trained Model

After training in this notebook, export the best model using **TorchScript** for
production inference. TorchScript serializes the model architecture + weights into
a single `.pt` file that can be loaded without the original class definitions.

```python
# Export EEGNet to TorchScript
model.eval()
dummy_input = torch.randn(1, 1, 64, 640).to(device)  # (batch, 1, channels, timepoints)
traced_model = torch.jit.trace(model, dummy_input)
traced_model.save('eegnet_bci_model.pt')
```

Alternatively, export to **ONNX** for broader compatibility (e.g., if you want to
run inference directly in C# using ONNX Runtime inside Unity):

```python
torch.onnx.export(
    model, dummy_input, 'eegnet_bci_model.onnx',
    input_names=['eeg_input'], output_names=['class_logits'],
    dynamic_axes={'eeg_input': {0: 'batch_size'}}
)
```

---

## 8.3 — Step 2: Python Real-Time Inference Server

The inference server is a Python script that:

1. **Connects to the EEG stream** via `pylsl.StreamInlet` (Lab Streaming Layer).
2. **Buffers incoming samples** into a sliding window (e.g., 4 seconds at 250 Hz = 1000 samples).
3. **Preprocesses** each window identically to training: band-pass 8–30 Hz, z-score normalization.
4. **Runs inference** through the TorchScript model.
5. **Sends the prediction** to Unity over a TCP socket as a JSON message.

```python
import pylsl
import socket
import json
import numpy as np
import torch
from scipy.signal import butter, sosfilt

# ── Load exported model ──
model = torch.jit.load('eegnet_bci_model.pt')
model.eval()

# ── Connect to EEG stream ──
streams = pylsl.resolve_stream('type', 'EEG')
inlet = pylsl.StreamInlet(streams[0])
sfreq = int(inlet.info().nominal_srate())  # e.g., 250 Hz

# ── Design band-pass filter (8-30 Hz) ──
sos = butter(5, [8, 30], btype='band', fs=sfreq, output='sos')

# ── TCP connection to Unity ──
unity_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
unity_socket.connect(('127.0.0.1', 5000))  # Unity listens on port 5000

# ── Sliding window buffer ──
WINDOW_SEC = 4.0  # seconds per classification window
STEP_SEC = 0.5    # advance by 0.5s between predictions
n_window = int(WINDOW_SEC * sfreq)  # 1000 samples
n_step = int(STEP_SEC * sfreq)      # 125 samples
buffer = np.zeros((64, n_window))    # (channels, timepoints)

while True:
    # Pull new samples from LSL stream
    chunk, _ = inlet.pull_chunk(max_samples=n_step)
    if len(chunk) == 0:
        continue
    chunk = np.array(chunk).T  # (channels, new_samples)

    # Shift buffer left and append new data
    buffer = np.roll(buffer, -chunk.shape[1], axis=1)
    buffer[:, -chunk.shape[1]:] = chunk

    # Preprocess: band-pass filter
    filtered = sosfilt(sos, buffer, axis=1)

    # Z-score normalize per channel
    mean = filtered.mean(axis=1, keepdims=True)
    std = filtered.std(axis=1, keepdims=True) + 1e-8
    normalized = (filtered - mean) / std

    # Inference
    x_tensor = torch.FloatTensor(normalized[np.newaxis, np.newaxis, :, :])
    with torch.no_grad():
        logits = model(x_tensor)
        probs = torch.softmax(logits, dim=1).numpy()[0]
        predicted_class = int(logits.argmax(dim=1).item())

    # Send to Unity
    message = json.dumps({
        'class': predicted_class,
        'confidence': float(probs[predicted_class]),
        'label': ['left_mi', 'right_mi', 'palmar_grasp', 'lateral_grasp'][predicted_class]
    }) + '\n'  # newline delimiter for Unity's StreamReader
    unity_socket.send(message.encode('utf-8'))
```

---

## 8.4 — Step 3: Unity Scene Setup

### 3a. Virtual Arm Model
- Import a rigged humanoid arm model (e.g., from Mixamo, Sketchfab, or Unity Asset Store).
- The model should have a proper **skeletal rig** with bones for: shoulder, upper arm, forearm,
  wrist, and individual fingers (at minimum thumb + index + ring for grasp distinction).
- Assign an **Animator Controller** with states: `Idle`, `ReachLeft`, `ReachRight`,
  `PalmarGrasp`, `LateralGrasp`, connected by transitions triggered by integer parameters.

### 3b. Animation Clips
- **ReachLeft / ReachRight**: shoulder flexion + elbow extension towards left/right targets.
- **PalmarGrasp**: all fingers curl around an object (power grip — e.g., grasping a cylinder).
- **LateralGrasp**: thumb + index finger pinch (key grip — e.g., turning a key).
- Use **Animation Blending** (blend trees) for smooth transitions between states.

### 3c. TCP Receiver Script (C#)

```csharp
using UnityEngine;
using System.Net.Sockets;
using System.IO;
using System.Threading;

[System.Serializable]
public class BCIPrediction
{
    public int classId;        // 0=left, 1=right, 2=palmar, 3=lateral
    public float confidence;   // softmax probability
    public string label;       // human-readable label
}

public class BCIReceiver : MonoBehaviour
{
    public string serverIP = "127.0.0.1";
    public int serverPort = 5000;
    public float confidenceThreshold = 0.6f;  // ignore low-confidence predictions
    public Animator armAnimator;               // reference to arm's Animator

    private TcpListener listener;
    private TcpClient client;
    private StreamReader reader;
    private BCIPrediction latestPrediction;
    private readonly object lockObj = new object();
    private bool isRunning = true;

    void Start()
    {
        // Start TCP listener on a background thread to avoid blocking Unity's main thread
        listener = new TcpListener(System.Net.IPAddress.Parse(serverIP), serverPort);
        listener.Start();
        Thread listenerThread = new Thread(ListenForData);
        listenerThread.IsBackground = true;  // thread dies when app exits
        listenerThread.Start();
    }

    void ListenForData()
    {
        client = listener.AcceptTcpClient();  // wait for Python to connect
        reader = new StreamReader(client.GetStream());

        while (isRunning)
        {
            string line = reader.ReadLine();
            if (line != null)
            {
                BCIPrediction pred = JsonUtility.FromJson<BCIPrediction>(line);
                lock (lockObj)
                {
                    latestPrediction = pred;  // thread-safe update
                }
            }
        }
    }

    void Update()
    {
        // Read latest prediction on Unity's main thread (required for Animator access)
        BCIPrediction pred;
        lock (lockObj)
        {
            pred = latestPrediction;
            latestPrediction = null;  // consume the prediction
        }

        if (pred != null && pred.confidence >= confidenceThreshold)
        {
            // Set Animator parameter to trigger the corresponding animation state
            armAnimator.SetInteger("BCIClass", pred.classId);
            Debug.Log($"BCI: {pred.label} (conf={pred.confidence:F2})");
        }
        else
        {
            // Below threshold → return to idle (prevents jittery switching)
            armAnimator.SetInteger("BCIClass", -1);
        }
    }

    void OnApplicationQuit()
    {
        isRunning = false;
        reader?.Close();
        client?.Close();
        listener?.Stop();
    }
}
```

---

## 8.5 — Step 4: Latency and Reliability Considerations

| Component | Typical Latency | How to Minimize |
|---|---|---|
| EEG acquisition | 4–16 ms (device buffer) | Use lowest hardware buffer size |
| LSL transport | < 1 ms (local) | Keep Python + Unity on same machine |
| Preprocessing | 2–5 ms | Pre-compute filter coefficients; avoid ICA at runtime |
| Model inference | 1–10 ms (CPU) / <1 ms (GPU) | Use TorchScript; batch size = 1 |
| TCP to Unity | < 1 ms (localhost) | Use `TcpClient` with `NoDelay = true` |
| Unity animation | 1 frame (16 ms @ 60 FPS) | Use Animator triggers, not coroutines |
| **Total** | **~25–50 ms** | Well below the 200 ms perceptual threshold |

**Reliability strategies:**
- **Confidence thresholding**: only act on predictions above 60% confidence to reduce false commands.
- **Temporal smoothing**: use a majority-vote buffer (e.g., 3 consecutive identical predictions) to filter noisy outputs.
- **Fallback to idle**: if no confident prediction arrives within 2 seconds, revert the arm to the idle state.
- **Visual feedback**: display a confidence bar in the Unity UI so the user can see when the BCI is uncertain.

---

## 8.6 — Step 5: Closed-Loop Neurofeedback for Rehabilitation

The key insight for stroke rehabilitation is that the virtual arm provides **real-time
visual neurofeedback**: when the patient imagines (or attempts) a grasp, they
immediately see the virtual arm execute it. This closes the sensorimotor loop that
is broken after stroke:

```
Motor intention  →  EEG signal  →  BCI classification  →  Virtual arm moves
       ↑                                                         │
       └─────────────── Visual cortex ◄──────────────────────────┘
                     (neurofeedback reinforcement)
```

**Rehabilitation protocol:**
1. **Calibration session** (~10 min): Record 40–60 trials per grasp type to train
   the subject-specific model.
2. **Training sessions** (20–30 min, 3×/week): Patient performs motor imagery while
   watching the virtual arm. The BCI provides immediate feedback.
3. **Adaptive difficulty**: Start with left/right MI (easier), progress to palmar/lateral
   grasp as the patient improves. Adjust confidence threshold dynamically.
4. **Progress tracking**: Log classification accuracy per session. Improving BCI accuracy
   over weeks correlates with cortical reorganization (neuroplasticity).

---

## 8.7 — Alternative: ONNX Runtime in Unity (No Python Server)

For a standalone deployment (no Python dependency), export the model to ONNX
(Section 8.2) and run inference directly inside Unity using the
[ONNX Runtime Unity Plugin](https://github.com/microsoft/onnxruntime/tree/main/csharp):

1. Install the `Microsoft.ML.OnnxRuntime` NuGet package in the Unity project.
2. Load the ONNX model in C#: `var session = new InferenceSession("eegnet_bci_model.onnx");`
3. Read EEG directly from the headset SDK (most vendors provide C# APIs).
4. Preprocess in C# (band-pass filter via `Math.NET Numerics`, z-score normalization).
5. Run inference: `var results = session.Run(inputs);`

This eliminates the Python server and TCP overhead, but requires reimplementing
the preprocessing pipeline in C#. It is recommended for final clinical deployment
where simplicity and portability matter.

---

## 8.8 — Summary

| Step | What | Tool |
|---|---|---|
| 1 | Export trained model | `torch.jit.trace()` or `torch.onnx.export()` |
| 2 | Real-time EEG streaming | Lab Streaming Layer (pylsl) |
| 3 | Python inference server | PyTorch + TCP socket |
| 4 | Unity virtual arm scene | Rigged model + Animator Controller |
| 5 | C# TCP receiver | `System.Net.Sockets.TcpListener` |
| 6 | Confidence thresholding | JSON `confidence` field |
| 7 | Neurofeedback protocol | Closed-loop motor imagery training |

The complete pipeline transforms offline EEG classification accuracy into a
real-time, interactive rehabilitation tool. The models trained in Sections 4–7
of this project provide the core intelligence; Unity provides the immersive
visual feedback that drives neuroplasticity in stroke patients.

In [ ]:
print("Section 8 complete -- Unity integration roadmap documented "
      "(model export → Python server → TCP → Unity virtual arm)")